In [2]:
import numpy as np
import matplotlib.pyplot as plt

In [3]:
X = np.linspace(0, 10, 50).reshape(-1, 1)

# X represents a single input feature that gradually increases from 0 to 10.
# This range is chosen so we can clearly see non-linear behavior


In [4]:

noise = np.random.normal(0, 0.2, size=(50, 1))
y = np.log(X + 1) + noise
# The logarithmic curve grows fast initially and then slows down.
# A straight line has only one slope and cannot match both behaviors at the same time.

In [5]:
# We choose 3 hidden units
hidden_units = 3

# Hidden units allow the model to combine multiple "pieces" of behavior.
# More than 1 helps bend the curve.
# Too many would make the model unstable and unnecessarily complex.

In [6]:

W1 = np.random.uniform(-1, 1, size=(1, hidden_units))
b1 = np.zeros((1, hidden_units))
W2 = np.random.uniform(-1, 1, size=(hidden_units, 1))
b2 = np.zeros((1, 1))

# W1 controls how input influences hidden layer.
# W2 controls how hidden layer influences output.
# Weights affect shape (bending).
# Biases shift the curve up or down.

In [7]:
def activation(z):
    return np.maximum(0, z)

def activation_slope(z):
    return (z > 0).astype(float)
# ReLU keeps positive values and suppresses negatives.
# Its slope is 1 for positive values and 0 for negative values.
# slope matters because slope tells the model how to adjust parameters — without slope, learning has no direction.

In [8]:
def forward(X):
    z1 = X @ W1 + b1
    h = activation(z1)
    y_hat = h @ W2 + b2
    return z1, h, y_hat

In [9]:
def compute_loss(y_hat, y):
    error = y_hat - y
    loss = np.mean(error ** 2)
    return error, loss
# Squaring amplifies large mistakes,
# forcing the model to correct them more strongly.
# Large error matters more because squaring makes big mistakes grow much larger than small ones.

In [10]:
def backward(X, y, z1, h, y_hat):
    N = len(X)

    error = y_hat - y
    dL_dy = 2 * error / N

    dL_dW2 = h.T @ dL_dy
    dL_db2 = np.sum(dL_dy, axis=0, keepdims=True)

    dL_dh = dL_dy @ W2.T
    dL_dz1 = dL_dh * activation_slope(z1)

    dL_dW1 = X.T @ dL_dz1
    dL_db1 = np.sum(dL_dz1, axis=0, keepdims=True)

    return dL_dW1, dL_db1, dL_dW2, dL_db2
#Q: How does activation control error flow?
#If activation slope is zero, error stops there; if slope is one, error flows backward normally.

In [11]:
learning_rate = 0.01

def update(dW1, db1, dW2, db2):
    global W1, b1, W2, b2

    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
# We move parameters opposite to the gradient,
# meaning we move opposite to the direction of increasing error.

In [12]:
epochs = 1000

for epoch in range(epochs):
    z1, h, y_hat = forward(X)
    error, loss = compute_loss(y_hat, y)
    dW1, db1, dW2, db2 = backward(X, y, z1, h, y_hat)
    update(dW1, db1, dW2, db2)

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss}")

Epoch 0, Loss: 3.009303600034758
Epoch 100, Loss: 0.07725262170810296
Epoch 200, Loss: 0.062451401036360335
Epoch 300, Loss: 0.060203044754046184
Epoch 400, Loss: 0.059873366661271736
Epoch 500, Loss: 0.05982586333181425
Epoch 600, Loss: 0.05981906787145083
Epoch 700, Loss: 0.05981809850287996
Epoch 800, Loss: 0.0598179603719251
Epoch 900, Loss: 0.059817940696883


Q: Why 1000 epochs?

Because bending a curve takes multiple gradual updates.

Q: Why learning rate 0.01?

Because it is small enough to stay stable but large enough to learn reasonably fast.